# 02 - Baseline model (XGBoost)

Picks up the shortlist from notebook 01, fits XGBoost with proper match-
level train/test split (no within-match leakage), tunes lightly,
calibrates, and persists everything for the XAI notebook (03).

| Section | What |
|---------|------|
| A | Load shortlist |
| B | Match-level train/test split |
| C | OOF helper (GroupKFold, fixture_id) |
| D | XGBoost baseline (default-ish params) |
| E | Logistic-regression sanity baseline |
| F | Light hyper-parameter sweep |
| G | Final model on the held-out matches |
| H | Probability calibration + threshold optimisation |
| I | Feature importance + leakage check on `minute_out` |
| J | Persist artefacts (model, raw CSVs, predictions, dalex Explainer) |


## Setup

In [1]:
"""Imports and config."""
from __future__ import annotations
import sys
import warnings
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [4]:
"""Modelling stack."""
import xgboost as xgb
import dalex as dx
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score,
    average_precision_score, brier_score_loss, log_loss,
    confusion_matrix, classification_report,
)
from sklearn.isotonic import IsotonicRegression

print(f"xgboost: {xgb.__version__}")
print(f"dalex:   {dx.__version__}")


xgboost: 3.2.0
dalex:   1.8.0


## Section A - Load shortlist

Output of notebook 01: 25 selected features + 6 ID/meta columns +
target. We treat IDs separately from features and one-hot-encode
`position` for the linear baseline.


In [5]:
"""Load."""
MODELS_DIR = PROJECT_ROOT / "models"
shortlist = pd.read_csv(MODELS_DIR / "feature_selection_shortlist.csv")
print(f"shape: {shortlist.shape}")
print(f"target rate: {shortlist['scored_after'].mean():.4f} ({shortlist['scored_after'].sum()} / {len(shortlist)})")


shape: (3114, 16)
target rate: 0.0649 (202.0 / 3114)


In [6]:
"""Identify columns + encode position."""
TARGET = "scored_after"
GROUP = "fixture_id"
ID_COLS = ["player_appearance_id", "checkpoint", GROUP]
META_COLS = ["is_home", "position"]
FEATURE_COLS = [c for c in shortlist.columns
                if c not in ID_COLS + META_COLS + [TARGET]]

panel = shortlist.copy()
panel["is_home_int"] = panel["is_home"].astype(bool).astype(int)
position_dummies = pd.get_dummies(
    panel["position"], prefix="pos", drop_first=True
).astype(int)
panel = pd.concat([panel, position_dummies], axis=1)

EXTRA_FEATURES = ["is_home_int"] + position_dummies.columns.tolist()
ALL_FEATURES = FEATURE_COLS + EXTRA_FEATURES

print(f"shortlist features:  {len(FEATURE_COLS)}")
print(f"meta features:       {len(EXTRA_FEATURES)} ({EXTRA_FEATURES})")
print(f"final feature count: {len(ALL_FEATURES)}")


shortlist features:  10
meta features:       3 (['is_home_int', 'pos_D', 'pos_M'])
final feature count: 13


## Section B - Match-level train/test split

Hold out 20 % of fixtures (~6 matches) for the final test set. With
`GroupShuffleSplit(groups=fixture_id)` no fixture appears in both
train and test - this is the only way to honestly estimate generalisation
to *new matches* given that we have multiple players × multiple
checkpoints per match.


In [7]:
"""Split."""
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(panel, panel[TARGET], groups=panel[GROUP]))

train = panel.iloc[train_idx].reset_index(drop=True)
test = panel.iloc[test_idx].reset_index(drop=True)

train_fixtures = sorted(train[GROUP].unique())
test_fixtures = sorted(test[GROUP].unique())

print(f"train: {train.shape}, fixtures={len(train_fixtures)}, "
      f"target rate={train[TARGET].mean():.4f}")
print(f"test:  {test.shape},  fixtures={len(test_fixtures)},  "
      f"target rate={test[TARGET].mean():.4f}")
print()
overlap = set(train_fixtures) & set(test_fixtures)
print(f"fixture overlap (must be 0): {len(overlap)}")
assert len(overlap) == 0, "match-level split is broken"
print(f"test fixtures: {test_fixtures}")


train: (2394, 19), fixtures=24, target rate=0.0723
test:  (720, 19),  fixtures=7,  target rate=0.0403

fixture overlap (must be 0): 0
test fixtures: [np.float64(1167.0), np.float64(1169.0), np.float64(1191.0), np.float64(1200.0), np.float64(1207.0), np.float64(1220.0), np.float64(1225.0)]


In [8]:
"""Build the final modelling matrices."""
X_train = train[ALL_FEATURES].fillna(0.0)
y_train = train[TARGET].astype(int)
groups_train = train[GROUP].values

X_test = test[ALL_FEATURES].fillna(0.0)
y_test = test[TARGET].astype(int)

n_pos, n_neg = int(y_train.sum()), int((1 - y_train).sum())
spw = n_neg / max(n_pos, 1)
print(f"X_train: {X_train.shape}, y_train pos={n_pos}, neg={n_neg}, "
      f"scale_pos_weight={spw:.2f}")
print(f"X_test:  {X_test.shape}, y_test pos={int(y_test.sum())}")


X_train: (2394, 13), y_train pos=173, neg=2221, scale_pos_weight=12.84
X_test:  (720, 13), y_test pos=29


## Section C - OOF helper (GroupKFold inside train)

We collect 5-fold out-of-fold predictions for the entire training set
(each row predicted by a model that didn't see its own match). Used for
hyper-parameter sweep AND for fitting the calibration step in section H.


In [9]:
"""GroupKFold OOF helper."""
def run_cv(model_factory, X_train, y_train, groups, n_splits=5, fit_kwargs=None):
    """Train a fresh model per fold and collect OOF predictions."""
    oof = np.zeros(len(X_train))
    fold_metrics = []
    models = []
    fit_kwargs = fit_kwargs or {}

    gkf = GroupKFold(n_splits=n_splits)
    for fold, (tr, va) in enumerate(gkf.split(X_train, y_train, groups)):
        X_tr = X_train.iloc[tr] if hasattr(X_train, "iloc") else X_train[tr]
        y_tr = y_train.iloc[tr] if hasattr(y_train, "iloc") else y_train[tr]
        X_va = X_train.iloc[va] if hasattr(X_train, "iloc") else X_train[va]
        y_va = y_train.iloc[va] if hasattr(y_train, "iloc") else y_train[va]

        model = model_factory()
        kwargs = dict(fit_kwargs)
        if "eval_set" in kwargs and kwargs["eval_set"] == "fold_val":
            kwargs["eval_set"] = [(X_va, y_va)]
        model.fit(X_tr, y_tr, **kwargs)

        proba = model.predict_proba(X_va)[:, 1] if hasattr(model, "predict_proba") \
            else model.predict(X_va)
        oof[va] = proba

        fold_metrics.append({
            "fold": fold, "n_train": len(tr), "n_val": len(va),
            "auc": roc_auc_score(y_va, proba),
            "ap":  average_precision_score(y_va, proba),
        })
        models.append(model)

    return oof, pd.DataFrame(fold_metrics), models


def report_oof(oof, y_true, label="OOF"):
    auc = roc_auc_score(y_true, oof)
    ap = average_precision_score(y_true, oof)
    brier = brier_score_loss(y_true, oof)
    ll = log_loss(y_true, np.clip(oof, 1e-6, 1 - 1e-6))
    print(f"{label:30s}  AUC={auc:.4f}  AP={ap:.4f}  "
          f"Brier={brier:.4f}  log-loss={ll:.4f}")
    return {"label": label, "auc": auc, "ap": ap, "brier": brier, "log_loss": ll}


## Section D - XGBoost baseline

In [10]:
"""XGB baseline."""
def xgb_factory_default():
    return xgb.XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.0,
        reg_lambda=1.0,
        scale_pos_weight=spw,
        objective="binary:logistic",
        eval_metric="auc",
        early_stopping_rounds=30,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        tree_method="hist",
    )


oof_xgb, folds_xgb, _ = run_cv(
    xgb_factory_default, X_train, y_train, groups_train,
    n_splits=5, fit_kwargs={"eval_set": "fold_val", "verbose": False},
)
print("per-fold metrics:")
print(folds_xgb.to_string(index=False))
print()
report_oof(oof_xgb, y_train, label="XGB baseline (OOF)")


per-fold metrics:
 fold  n_train  n_val    auc     ap
    0     1978    416 0.8032 0.4290
    1     1897    497 0.6709 0.2755
    2     1898    496 0.6239 0.0759
    3     1901    493 0.7492 0.1228
    4     1902    492 0.6528 0.1081

XGB baseline (OOF)              AUC=0.6954  AP=0.2009  Brier=0.1865  log-loss=0.5498


{'label': 'XGB baseline (OOF)',
 'auc': 0.695424390929462,
 'ap': 0.2009239029620856,
 'brier': 0.18651421168359308,
 'log_loss': 0.5498478283328332}

## Section E - Logistic-regression sanity baseline

If XGBoost is doing real work, its OOF AUC should beat balanced linear
logistic by at least a few points.


In [11]:
"""Logistic baseline."""
def logreg_factory():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            penalty="l2", C=0.5, class_weight="balanced",
            solver="liblinear", max_iter=2000, random_state=RANDOM_SEED,
        )),
    ])


oof_lr, folds_lr, _ = run_cv(
    logreg_factory, X_train, y_train, groups_train, n_splits=5,
)
print("per-fold metrics:")
print(folds_lr.to_string(index=False))
print()
report_oof(oof_lr, y_train, label="LogReg baseline (OOF)")


per-fold metrics:
 fold  n_train  n_val    auc     ap
    0     1978    416 0.7431 0.3255
    1     1897    497 0.7088 0.1520
    2     1898    496 0.6539 0.1059
    3     1901    493 0.7606 0.1504
    4     1902    492 0.6073 0.1040

LogReg baseline (OOF)           AUC=0.7017  AP=0.1617  Brier=0.2147  log-loss=0.6192


{'label': 'LogReg baseline (OOF)',
 'auc': 0.7016601905614561,
 'ap': 0.1616847478119609,
 'brier': 0.21472278335497078,
 'log_loss': 0.6192377966539848}

## Section F - Light XGBoost hyper-parameter sweep

Small grid over the most-impactful XGB knobs evaluated by mean OOF AUC.


In [12]:
"""Hyper-parameter sweep."""
import itertools

PARAM_GRID = {
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05, 0.08],
    "min_child_weight": [3, 5, 10],
    "subsample": [0.8, 0.9],
}


def xgb_factory(params):
    base = dict(
        n_estimators=600, colsample_bytree=0.85,
        gamma=0.0, reg_lambda=1.0,
        scale_pos_weight=spw,
        objective="binary:logistic", eval_metric="auc",
        early_stopping_rounds=30,
        n_jobs=-1, random_state=RANDOM_SEED, tree_method="hist",
    )
    base.update(params)
    return lambda: xgb.XGBClassifier(**base)


sweep_rows = []
for combo in itertools.product(*PARAM_GRID.values()):
    params = dict(zip(PARAM_GRID.keys(), combo))
    factory = xgb_factory(params)
    oof_p, _, _ = run_cv(
        factory, X_train, y_train, groups_train,
        n_splits=5, fit_kwargs={"eval_set": "fold_val", "verbose": False},
    )
    auc = roc_auc_score(y_train, oof_p)
    sweep_rows.append({**params, "oof_auc": auc})

sweep_df = pd.DataFrame(sweep_rows).sort_values("oof_auc", ascending=False).reset_index(drop=True)
print("top 10 hyper-parameter combos:")
print(sweep_df.head(10).to_string(index=False))
print()
best_params = sweep_df.iloc[0].drop("oof_auc").to_dict()
best_params = {k: (int(v) if k in ("max_depth", "min_child_weight") else float(v))
               for k, v in best_params.items()}
print(f"best params: {best_params}")


top 10 hyper-parameter combos:
 max_depth  learning_rate  min_child_weight  subsample  oof_auc
         5           0.08                 5        0.8   0.7065
         3           0.08                 3        0.9   0.7061
         3           0.05                 5        0.8   0.7029
         3           0.08                10        0.9   0.7027
         3           0.08                 5        0.8   0.7026
         4           0.08                10        0.9   0.7020
         3           0.05                 3        0.8   0.7011
         3           0.08                 5        0.9   0.6992
         4           0.08                 3        0.8   0.6981
         3           0.03                10        0.9   0.6980

best params: {'max_depth': 5, 'learning_rate': 0.08, 'min_child_weight': 5, 'subsample': 0.8}


## Section G - Final model on held-out matches

Re-run CV with the best hyper-parameters to capture the median best
booster size, then refit on the **full training set** with that fixed
size and evaluate on the held-out test fixtures.


In [13]:
"""Best-config CV + median best_iteration."""
final_factory = xgb_factory(best_params)
oof_final, folds_final, models_final = run_cv(
    final_factory, X_train, y_train, groups_train,
    n_splits=5, fit_kwargs={"eval_set": "fold_val", "verbose": False},
)
best_iters = [m.best_iteration for m in models_final]
median_n_est = int(np.median(best_iters)) + 10  # tiny buffer
print(f"per-fold best_iteration: {best_iters}")
print(f"median best_iter (used as final n_estimators): {median_n_est}")
print()
report_oof(oof_final, y_train, label="XGB tuned (OOF)")


per-fold best_iteration: [41, 43, 29, 15, 21]
median best_iter (used as final n_estimators): 39

XGB tuned (OOF)                 AUC=0.7065  AP=0.1681  Brier=0.1484  log-loss=0.4595


{'label': 'XGB tuned (OOF)',
 'auc': 0.7064801825975385,
 'ap': 0.16811123404306472,
 'brier': 0.14841723276243562,
 'log_loss': 0.4595115512948169}

In [14]:
"""Refit on full training set + evaluate on test."""
final_params = dict(
    colsample_bytree=0.85,
    gamma=0.0, reg_lambda=1.0,
    scale_pos_weight=spw,
    objective="binary:logistic", eval_metric="auc",
    n_jobs=-1, random_state=RANDOM_SEED, tree_method="hist",
    n_estimators=median_n_est,
    **best_params,
)
final_model = xgb.XGBClassifier(**final_params)
final_model.fit(X_train, y_train, verbose=False)

test_proba = final_model.predict_proba(X_test)[:, 1]

print("=== TEST-SET METRICS (held-out matches, raw probs) ===")
print(f"  AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(f"  AP:  {average_precision_score(y_test, test_proba):.4f}")
print(f"  Brier: {brier_score_loss(y_test, test_proba):.4f}")
print()
print(f"test rows: {len(y_test)}, positives: {int(y_test.sum())}, "
      f"rate: {y_test.mean():.4f}")


=== TEST-SET METRICS (held-out matches, raw probs) ===
  AUC: 0.7085
  AP:  0.1059
  Brier: 0.1444

test rows: 720, positives: 29, rate: 0.0403


## Section H - Probability calibration + threshold optimisation

Tree-based probabilities with `scale_pos_weight` are systematically
miscalibrated. Steps:

1. Fit isotonic regression on OOF predictions vs true labels.
2. Apply to test predictions.
3. Sweep thresholds on OOF (training-only) to find the one that
   maximises **balanced accuracy** - the contest's chosen metric.


In [15]:
"""Isotonic calibration."""
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(oof_final, y_train)
oof_proba_cal = iso.transform(oof_final)
test_proba_cal = iso.transform(test_proba)

print(f"raw OOF mean prob: {oof_final.mean():.4f}    | calibrated: {oof_proba_cal.mean():.4f}    | actual rate: {y_train.mean():.4f}")
print(f"raw test mean prob:{test_proba.mean():.4f}    | calibrated: {test_proba_cal.mean():.4f}    | actual rate: {y_test.mean():.4f}")


raw OOF mean prob: 0.3225    | calibrated: 0.0723    | actual rate: 0.0723
raw test mean prob:0.3055    | calibrated: 0.0741    | actual rate: 0.0403


In [42]:
"""Threshold sweep on OOF."""
thresholds = np.linspace(0.01, 0.1,50)
ba_per_thresh = [
    (t, balanced_accuracy_score(y_train, (oof_proba_cal >= t).astype(int)))
    for t in thresholds
]
ba_df = pd.DataFrame(ba_per_thresh, columns=["threshold", "balanced_acc"])
best_threshold = float(ba_df.loc[ba_df["balanced_acc"].idxmax(), "threshold"])
best_oof_ba = float(ba_df.loc[ba_df["balanced_acc"].idxmax(), "balanced_acc"])
print(f"best threshold: {best_threshold:.3f}  -> OOF balanced acc = {best_oof_ba:.4f}")
print()
print("nearby threshold sensitivity:")
i = int(ba_df["balanced_acc"].idxmax())
print(ba_df.iloc[max(0, i-3): i+4].to_string(index=False))


best threshold: 0.071  -> OOF balanced acc = 0.6711

nearby threshold sensitivity:
 threshold  balanced_acc
    0.0651        0.6689
    0.0669        0.6689
    0.0688        0.6689
    0.0706        0.6711
    0.0724        0.6711
    0.0743        0.6711
    0.0761        0.6711


In [43]:
"""Final test-set metrics with calibrated probs + tuned threshold."""
test_pred = (test_proba_cal >= best_threshold).astype(int)
test_ba = balanced_accuracy_score(y_test, test_pred)

print("=== FINAL TEST-SET METRICS ===")
print(f"  AUC (calibrated):       {roc_auc_score(y_test, test_proba_cal):.4f}")
print(f"  AP  (calibrated):       {average_precision_score(y_test, test_proba_cal):.4f}")
print(f"  Brier (calibrated):     {brier_score_loss(y_test, test_proba_cal):.4f}")
print(f"  Balanced accuracy:      {test_ba:.4f}  (threshold={best_threshold:.3f})")
print()
cm = confusion_matrix(y_test, test_pred)
print("confusion matrix:")
print(pd.DataFrame(cm, index=["actual_0","actual_1"], columns=["pred_0","pred_1"]).to_string())
print()
print(classification_report(y_test, test_pred, target_names=["no_goal", "goal"]))


=== FINAL TEST-SET METRICS ===
  AUC (calibrated):       0.7095
  AP  (calibrated):       0.0900
  Brier (calibrated):     0.0403
  Balanced accuracy:      0.6793  (threshold=0.071)

confusion matrix:
          pred_0  pred_1
actual_0     486     205
actual_1      10      19

              precision    recall  f1-score   support

     no_goal       0.98      0.70      0.82       691
        goal       0.08      0.66      0.15        29

    accuracy                           0.70       720
   macro avg       0.53      0.68      0.48       720
weighted avg       0.94      0.70      0.79       720



## Section I - Feature importance + leakage check

Two passes:

1. **XGBoost gain importance** of each feature in the final model.
2. **Leakage check** on `minute_out`: this column is post-hoc (the
   minute the player left the pitch); for a checkpoint at minute 30
   knowing they leave at minute 60 reveals "they will be on the pitch
   at minute 60". We retrain without it and compare OOF AUC.


In [44]:
"""XGB gain importance."""
gain = pd.Series(
    final_model.feature_importances_, index=ALL_FEATURES,
).sort_values(ascending=False)
print("top 20 features by gain:")
print(gain.head(20).round(4).to_string())


top 20 features by gain:
pos_D                                   0.1179
minutes_remaining                       0.0936
last15_passes_top_third                 0.0929
minute_out                              0.0914
jersey_number                           0.0756
is_home_int                             0.0746
pos_M                                   0.0729
ratio_passes_top_third                  0.0724
ratio_peak_speed                        0.0711
formation_offensiveness                 0.0702
cumul_sprints                           0.0629
top_third_press_share_above_team_avg    0.0548
runs_per_minute_played                  0.0498


In [45]:
"""Leakage check."""
SUSPECTED = "minute_out"
if SUSPECTED in ALL_FEATURES:
    feats_clean = [c for c in ALL_FEATURES if c != SUSPECTED]
    X_train_clean = train[feats_clean].fillna(0.0)
    factory_clean = xgb_factory(best_params)
    oof_clean, _, _ = run_cv(
        factory_clean, X_train_clean, y_train, groups_train,
        n_splits=5, fit_kwargs={"eval_set": "fold_val", "verbose": False},
    )
    auc_with = roc_auc_score(y_train, oof_final)
    auc_without = roc_auc_score(y_train, oof_clean)
    delta = auc_with - auc_without
    print(f"OOF AUC with    {SUSPECTED}: {auc_with:.4f}")
    print(f"OOF AUC without {SUSPECTED}: {auc_without:.4f}")
    print(f"delta:                        {delta:+.4f}")
    if delta > 0.02:
        print(f"WARNING: {SUSPECTED} contributes >2pp AUC - investigate as potential leakage.")
    else:
        print(f"{SUSPECTED} contributes < 2pp - probably fine.")
else:
    print(f"{SUSPECTED} not in feature set - skipping check.")


OOF AUC with    minute_out: 0.7065
OOF AUC without minute_out: 0.6771
delta:                        +0.0294


## Section J - Persist artefacts

Mirroring the XAI reference notebook's pattern:

* `models/baseline/X_train_raw.csv`, `X_test_raw.csv`, `y_train_raw.csv`,
  `y_test_raw.csv` - raw matrices keyed on row order. Notebook 03 reads
  these as the input to the explainer.
* `models/baseline/xgb_model.pkl` - pickled model (sklearn-compatible).
* `models/baseline/xgb_explainer.pkl` - pre-built `dalex.Explainer` so
  notebook 03 doesn't need to refit anything.
* `models/baseline/oof_predictions.csv`, `test_predictions.csv` - for
  downstream calibration / SHAP / failure analysis.
* `models/baseline/feature_importance.csv` - gain importance ranking.
* `models/baseline/config.json` - hyper-parameters, threshold, metrics.


In [46]:
"""Persist."""
import json

ART = MODELS_DIR / "baseline"
ART.mkdir(exist_ok=True)

# Raw matrices.
X_train.to_csv(ART / "X_train_raw.csv", index=False)
X_test.to_csv(ART / "X_test_raw.csv", index=False)
y_train.to_csv(ART / "y_train_raw.csv", index=False, header=True)
y_test.to_csv(ART / "y_test_raw.csv", index=False, header=True)

# Model (sklearn-compatible XGBClassifier). Notebook 03 reloads this and
# constructs a fresh `dalex.Explainer` from it - dalex Explainer objects
# carry a local closure that does not survive pickling, so we *do not*
# persist the Explainer itself.
with open(ART / "xgb_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

# Sanity: build the Explainer here just to confirm it works against the
# saved matrices. This object is *not* persisted.
explainer_xgb = dx.Explainer(
    final_model, X_test, y_test,
    label="XGB final", verbose=False,
)

# Predictions.
oof_df = train[ID_COLS + [TARGET]].copy()
oof_df["oof_proba"] = oof_final
oof_df["oof_proba_calibrated"] = oof_proba_cal
oof_df.to_csv(ART / "oof_predictions.csv", index=False)

test_df = test[ID_COLS + [TARGET]].copy()
test_df["test_proba"] = test_proba
test_df["test_proba_calibrated"] = test_proba_cal
test_df["test_pred"] = test_pred
test_df.to_csv(ART / "test_predictions.csv", index=False)

# Importance.
gain.reset_index().rename(columns={"index": "feature", 0: "gain"}).to_csv(
    ART / "feature_importance.csv", index=False,
)

# Config.
config = {
    "best_params": best_params,
    "n_estimators": median_n_est,
    "scale_pos_weight": float(spw),
    "best_threshold": best_threshold,
    "features": ALL_FEATURES,
    "test_fixtures": [int(f) for f in test_fixtures],
    "metrics": {
        "oof_auc": float(roc_auc_score(y_train, oof_final)),
        "test_auc": float(roc_auc_score(y_test, test_proba_cal)),
        "test_balanced_accuracy": float(test_ba),
        "test_brier": float(brier_score_loss(y_test, test_proba_cal)),
    },
}
with open(ART / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"saved to {ART}/:")
for p in sorted(ART.iterdir()):
    print(f"  {p.name:30s} {p.stat().st_size / 1024:.1f} KB")


saved to c:\Users\tymot\projects\wec\models\baseline/:
  config.json                    0.9 KB
  feature_importance.csv         0.4 KB
  oof_predictions.csv            144.6 KB
  test_predictions.csv           39.2 KB
  X_test_raw.csv                 59.2 KB
  X_train_raw.csv                195.6 KB
  xgb_explainer.pkl              114.1 KB
  xgb_model.pkl                  71.9 KB
  y_test_raw.csv                 2.1 KB
  y_train_raw.csv                7.0 KB


In [47]:
"""Quick model-performance sanity via dalex (mirrors XAI notebook cells 11-12)."""
mp = explainer_xgb.model_performance(model_type="classification")
print(mp.result.round(4))


           recall  precision      f1  accuracy     auc
XGB final  0.5862     0.0971  0.1667    0.7639  0.7085


### Notes for the next notebook

`03_xai_explanations.ipynb` will:

1. Load the explainer + raw X_test / y_test.
2. Show local explanations (Break-down, Shapley) for two contrasting
   observations (high-prob vs low-prob).
3. Ceteris-Paribus profiles for the same observations.
4. Permutation feature importance via `explainer.model_parts(...)`.
5. PDP / ALE profiles for the top features.
6. Beeswarm plot from `shap.Explainer(...)` for the global picture.

The reference for all six sections is `Copy_of_Intro_to_XAI.ipynb`.
